# Dự báo vỡ nợ tín dụng — pipeline đầy đủ từ dữ liệu thô đến đánh giá và so sánh mô hình

Notebook này chạy **toàn bộ** quy trình của đồ án trong một file, **không phụ thuộc vào thư mục `src/`**, để có thể đọc từ trên xuống dưới và hiểu từng bước:

```
data/raw (dữ liệu gốc)
   │
   ▼
[1] Tầng A — Khai báo dữ liệu      đọc file gốc, mã hóa nhãn 1 = xấu, bỏ cột ID / rò rỉ, mã đặc biệt → NaN
   │                                (không học gì từ dữ liệu → được phép làm trước khi chia)
   ▼
[2] Khám phá dữ liệu nhanh (EDA)   tỷ lệ vỡ nợ, tỷ lệ thiếu, rò rỉ qua giá trị thiếu ở HMEQ
   │
   ▼
[3] Chia dữ liệu                   5-fold StratifiedGroupKFold (nhóm = dòng trùng lặp);
   │                                Lending Club: chia theo thời gian (train 2013, val 2014, test 2015)
   ▼
[4] Tầng B.0 — Sàng lọc cột        bỏ cột rỗng / hằng số — fit CHỈ trên train của từng fold
   │
   ▼
[5] Tầng B.1 — Tiền xử lý Baesens  điền median, one-hot / mã số nguyên, chuẩn hóa — fit CHỈ trên train
   │
   ▼
[6–8] Huấn luyện 7 mô hình          LR, KNN, RF, NB (classic) + XGBoost, LightGBM, CatBoost (GBDT)
   │                                tham số mặc định; ngưỡng quyết định chọn trên validation
   ▼
[9–13] Đánh giá và so sánh          AUC, Gini, KS, Brier, LogLoss, AP, Precision/Recall/F1;
                                    xếp hạng, Friedman, Nemenyi (CD), Wilcoxon–Holm, bootstrap
```

Giao thức tiền xử lý và chia dữ liệu theo **Baesens et al. (2026)** (chi tiết trong `configs/preprocessing_v1.md`).

### Bảy bộ dữ liệu chính

| Bộ dữ liệu | Số dòng dùng | Đặc điểm |
|---|---:|---|
| Give Me Some Credit (GMSC) | 150.000 | 10 biến số, có giá trị 96/98 bất thường |
| Taiwan Default Credit | 30.000 | lịch sử trả nợ 6 tháng |
| Home Credit | 307.511 | 120 cột, nhiều giá trị thiếu, có biến chuỗi |
| HMEQ | 5.960 | nhỏ; **mẫu hình thiếu dữ liệu tiết lộ nhãn** |
| South German Credit | 1.000 | bản sửa lỗi của German Credit, hầu hết là biến phân loại |
| FICO HELOC | 9.871 | mã đặc biệt −7/−8/−9 |
| Lending Club | 545.779 | chia theo thời gian, dữ liệu thô ~1,6 GB |

Thêm **HMEQ (che giá trị thiếu)** — `hmeq_blind` — làm phân tích phụ về rò rỉ.

### Cách chạy

* Chạy toàn bộ: *Kernel → Restart & Run All*. Thời gian khoảng **30–45 phút** trên laptop (Home Credit và Lending Club chiếm phần lớn); cần khoảng 8 GB RAM.
* Chạy thử nhanh (chỉ 4 bộ nhỏ, ~2 phút): đặt biến môi trường `NB_FAST=1` trước khi mở Jupyter, hoặc sửa `CHE_DO_NHANH = True` ở ô cấu hình.
* Kết quả (bảng CSV, dự đoán, hình) được lưu vào `reports/notebook_full/`. Notebook **không ghi đè** `data/splits`, `data/processed` hay `reports/models` của pipeline chính.

## 0. Thiết lập môi trường

**Lưu ý quan trọng:** ở đây **chưa** import `xgboost`, `lightgbm`, `catboost`. Trên macOS, hai thư viện XGBoost/LightGBM mang theo thư viện OpenMP riêng; nếu chúng được nạp cùng tiến trình với KNN đa luồng của scikit-learn thì kernel có thể bị treo hoặc crash. Vì vậy notebook huấn luyện **toàn bộ mô hình classic trước** (mục 7), rồi mới import và huấn luyện GBDT (mục 8).

In [ ]:
# --- Thư viện chuẩn của Python ---
import os
import re
import json
import time
import itertools
import warnings
from pathlib import Path

# --- Xử lý dữ liệu dạng bảng ---
import numpy as np
import pandas as pd
import pyarrow as pa            # đọc file CSV lớn (Lending Club) nhanh và tiết kiệm bộ nhớ
import pyarrow.csv as pacsv
import pyarrow.compute as pc

# --- Vẽ hình và thống kê ---
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy import stats

# --- scikit-learn: chia dữ liệu, tiền xử lý, mô hình, độ đo ---
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, f1_score,
                             log_loss, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)

# Hiển thị bảng rộng hơn và hình đẹp hơn
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
# ===================== CẤU HÌNH CHUNG =====================

# Thư mục gốc của project = thư mục cha gần nhất có chứa data/raw
# (notebook có thể được mở từ notebooks/ hoặc từ thư mục gốc)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"                       # dữ liệu gốc, chỉ đọc
OUT = ROOT / "reports" / "notebook_full"          # nơi lưu kết quả của notebook này
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
(OUT / "predictions").mkdir(exist_ok=True)

SEED = 42                  # cố định hạt giống ngẫu nhiên → chạy lại ra đúng kết quả cũ
N_FOLDS = 5                # 5-fold cross-validation (Baesens et al. 2026, mục 4.3)
MISSING_SHIFT_THR = 0.30   # B.0: cảnh báo nếu tỷ lệ thiếu train và val/test lệch > 30 điểm %
LC_START_YEAR = 2013       # Lending Club: năm X — trước 2013 thiếu ~22 trường bureau (xem mục 1.7)
N_BOOTSTRAP = 1000         # số lần lặp bootstrap cho Lending Club (chỉ có 1 lần chia)

# Chế độ nhanh: chỉ chạy 4 bộ dữ liệu nhỏ để kiểm tra code (khoảng 2 phút)
CHE_DO_NHANH = os.environ.get("NB_FAST", "0") == "1"

# Dùng lại kết quả huấn luyện đã lưu ở reports/notebook_full (bỏ qua mục 7–8, tiết kiệm ~30 phút)
DUNG_KET_QUA_CU = os.environ.get("NB_REUSE", "0") == "1"

TAT_CA = ["give_me_some_credit", "taiwan_default_credit", "home_credit", "hmeq",
          "south_german_credit", "fico_heloc", "lending_club_B", "hmeq_blind"]
NHO = ["taiwan_default_credit", "hmeq", "south_german_credit", "fico_heloc", "hmeq_blind"]
DATASETS = NHO if CHE_DO_NHANH else TAT_CA

# 7 bộ dùng cho so sánh chính (hmeq_blind chỉ là phân tích phụ về rò rỉ)
MAIN_DATASETS = [d for d in ["give_me_some_credit", "taiwan_default_credit", "home_credit", "hmeq",
                             "south_german_credit", "fico_heloc", "lending_club_B"] if d in DATASETS]

# Tên ngắn để vẽ hình
LABELS = {"give_me_some_credit": "GMSC", "taiwan_default_credit": "Taiwan", "home_credit": "Home Credit",
          "hmeq": "HMEQ", "hmeq_blind": "HMEQ (che thiếu)", "south_german_credit": "South German",
          "fico_heloc": "HELOC", "lending_club_B": "Lending Club"}

CLASSIC_MODELS = ["LR", "KNN", "RF", "NB"]
GBDT_MODELS = ["XGB", "LGBM", "CatBoost"]
MODELS = CLASSIC_MODELS + GBDT_MODELS
COLORS = {"LR": "#4C72B0", "KNN": "#DD8452", "RF": "#55A868", "NB": "#C44E52",
          "XGB": "#8172B3", "LGBM": "#937860", "CatBoost": "#DA8BC3"}

print("Thư mục gốc :", ROOT)
print("Kết quả lưu :", OUT)
print("Chế độ nhanh:", CHE_DO_NHANH, "| dùng kết quả cũ:", DUNG_KET_QUA_CU)
print("Bộ dữ liệu  :", DATASETS)

## 1. Tầng A — Khai báo dữ liệu

Tầng A chỉ làm những việc **không học từ dữ liệu**:

| # | Thao tác | Ví dụ |
|---|---|---|
| A1 | Chọn đúng bảng có nhãn | Home Credit: chỉ `application_train.csv` |
| A2 | Lọc dòng theo quy tắc cố định | Lending Club: chỉ khoản vay đã kết thúc |
| A3 | Mã hóa nhãn: **1 = vỡ nợ (xấu)**, 0 = tốt | South German gốc: 1 = tốt → đảo lại |
| A5–A6 | Bỏ cột ID, văn bản, cột **rò rỉ** (chỉ biết sau khi cho vay) | `total_pymnt`, `recoveries` của Lending Club |
| A8 | Mã đặc biệt → NaN | HELOC: −7/−8/−9; Home Credit: `DAYS_EMPLOYED = 365243` |
| A9 | Đổi định dạng | `"Aug-2003"` → 2003,58 |
| A10 | Khai báo biến số / biến phân loại | Taiwan: `SEX`, `EDUCATION`, `MARRIAGE` |

**Phép thử một dòng:** nếu một bước cho đúng kết quả khi chỉ đưa vào *một dòng duy nhất*, bước đó không học từ dữ liệu → được phép chạy trên toàn bộ dữ liệu trước khi chia. Ngược lại (median, trung bình, danh sách nhóm…) phải fit **trong từng fold, chỉ trên train** — nếu không sẽ **rò rỉ** thông tin từ tập test vào mô hình và cho kết quả đẹp giả tạo.

Mỗi bộ dữ liệu sau tầng A có: `row_id` (số thứ tự dòng trong file gốc), các cột feature, cột `target`.

### 1.1. Hàm đọc file dùng chung

Bốn bộ (HMEQ, HELOC, Taiwan, GMSC) được lưu ở định dạng **ARFF** (định dạng của Weka): phần đầu khai báo `@ATTRIBUTE <tên> <kiểu>`, sau dòng `@DATA` là dữ liệu dạng CSV, giá trị thiếu ghi là `?`.

In [ ]:
def read_arff(path: Path) -> pd.DataFrame:
    """Đọc file ARFF tối giản: lấy tên cột từ các dòng @attribute, phần dữ liệu đọc như CSV, '?' = NaN."""
    names, data_line = [], None
    with open(path, encoding="utf-8", errors="replace") as fh:
        for i, line in enumerate(fh):
            s = line.strip()
            low = s.lower()
            if low.startswith("@attribute"):
                # tên cột có thể nằm trong '...' hoặc "..." hoặc không có dấu nháy
                m = re.match(r"@attribute\s+('([^']*)'|\"([^\"]*)\"|(\S+))", s, re.I)
                names.append(m.group(2) or m.group(3) or m.group(4))
            elif low.startswith("@data"):
                data_line = i + 1          # dữ liệu bắt đầu ngay sau dòng @data
                break
    df = pd.read_csv(path, skiprows=data_line, header=None, names=names, na_values=["?"],
                     quotechar="'", skipinitialspace=True, comment="%", skip_blank_lines=True)
    for c in df.select_dtypes("object"):   # giá trị chuỗi đôi khi có khoảng trắng thừa
        df[c] = df[c].str.strip()
    return df


def finish(df: pd.DataFrame, label: str, target: pd.Series) -> pd.DataFrame:
    """A3: thay cột nhãn gốc bằng cột `target` (1 = xấu, 0 = tốt).
    Cột nhãn gốc bị xóa để không thể vô tình lọt vào feature."""
    df = df.drop(columns=[label])
    df["target"] = target.astype("int8")
    return df


def with_row_id(df: pd.DataFrame) -> pd.DataFrame:
    """Thêm cột row_id = vị trí dòng trong file gốc — khóa ổn định để ghi nhận dòng nào thuộc fold nào."""
    df = df.reset_index(drop=True)
    df.insert(0, "row_id", np.arange(len(df), dtype=np.int64))
    return df

### 1.2. Bốn bộ dữ liệu nhỏ dạng ARFF và South German

* **GMSC** — nhãn `SeriousDlqin2yrs` (đã là 1 = xấu). Giá trị 96/98 ở các cột đếm số lần trễ hạn là mã bất thường nhưng **giữ nguyên** để đúng với Baesens.
* **Taiwan** — nhãn `default.payment.next.month`; bỏ cột `ID`; `SEX`, `EDUCATION`, `MARRIAGE` là biến phân loại; `PAY_0…PAY_6` (số tháng trễ) giữ là biến số.
* **HMEQ** — nhãn `BAD`; `REASON`, `JOB` là biến phân loại.
* **HELOC** — nhãn `RiskPerformance`: `"Bad"` → 1. Các mã −7 (không có điều kiện), −8 (không có giao dịch hợp lệ), −9 (không có bản ghi) được đổi thành NaN vì chúng **không phải giá trị số thật**.
* **South German Credit** — bản sửa lỗi mã hóa của German Credit (Grömping 2019). Trong codetable `credit_risk`: 1 = tốt, 0 = xấu → `target = 1 − credit_risk`. Chỉ 3 biến số thật (`duration`, `amount`, `age`), còn lại là mã nhóm → khai báo là phân loại.

In [ ]:
def load_give_me_some_credit():
    df = read_arff(RAW / "give_me_some_credit")
    return with_row_id(finish(df, "SeriousDlqin2yrs", df["SeriousDlqin2yrs"]))


def load_taiwan():
    df = read_arff(RAW / "taiwan_default_credit")
    # cột tiền (LIMIT_BAL, BILL_AMT*, PAY_AMT*) là số thực; các cột còn lại là số nguyên
    float_cols = [c for c in df.columns if c.startswith(("LIMIT_BAL", "BILL_AMT", "PAY_AMT"))]
    int_cols = [c for c in df.columns if c not in float_cols]
    df[int_cols] = df[int_cols].astype(int)
    label = "default.payment.next.month"
    return with_row_id(finish(df, label, df[label])).drop(columns="ID")   # A5: bỏ ID


def load_hmeq():
    df = read_arff(RAW / "HMEQ")
    return with_row_id(finish(df, "BAD", df["BAD"]))


def load_fico_heloc():
    df = read_arff(RAW / "fico_heloc")
    df = with_row_id(finish(df, "RiskPerformance", (df["RiskPerformance"] == "Bad").astype(int)))
    feats = df.columns.drop(["row_id", "target"])
    df[feats] = df[feats].mask(df[feats].isin([-7, -8, -9]))              # A8: mã đặc biệt -> NaN
    return df


SOUTH_GERMAN_COLUMNS = [  # tên tiếng Anh theo codetable.txt (file gốc dùng tên tiếng Đức)
    "status", "duration", "credit_history", "purpose", "amount", "savings",
    "employment_duration", "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property", "age",
    "other_installment_plans", "housing", "number_credits", "job",
    "people_liable", "telephone", "foreign_worker", "credit_risk",
]


def load_south_german_credit():
    df = pd.read_csv(RAW / "south_german_credit" / "SouthGermanCredit.asc", sep=r"\s+")
    df.columns = SOUTH_GERMAN_COLUMNS
    return with_row_id(finish(df, "credit_risk", 1 - df["credit_risk"]))  # 0 = xấu -> 1 = xấu

### 1.3. Home Credit

Chỉ dùng bảng chính `application_train.csv` (307.511 hồ sơ). Các bảng phụ (`bureau`, `previous_application`…) cần tổng hợp — đó là feature engineering, không thuộc giao thức Baesens.

* `SK_ID_CURR` là mã hồ sơ: **giữ làm khóa nhưng không phải feature**.
* `DAYS_EMPLOYED = 365243` (≈ 1.000 năm) là mã cho "không đi làm / hưu trí" → NaN.
* Mọi cột kiểu chuỗi (`NAME_*`, `CODE_GENDER`, `OCCUPATION_TYPE`…) được coi là biến phân loại.

In [ ]:
def load_home_credit():
    df = pd.read_csv(RAW / "home_credit" / "application_train.csv")
    df = with_row_id(finish(df, "TARGET", df["TARGET"]))
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].mask(df["DAYS_EMPLOYED"] == 365243)   # A8
    return df

### 1.4. Lending Club — khai báo chi tiết

File `accepted_2007_to_2018Q4.csv.gz` có ~2,26 triệu khoản vay × 151 cột (~1,6 GB khi giải nén). Để tiết kiệm RAM, file được đọc **theo từng khối** và lọc ngay:

**Lọc dòng (A2), mỗi bộ lọc là điều kiện cố định trên từng dòng:**

1. `application_type == "Individual"` — hồ sơ đồng vay chỉ có từ ~2015, gây dịch chuyển theo thời gian.
2. `term == 36 tháng` — khoản 60 tháng đáo hạn chậm, nhiều khoản chưa kết thúc.
3. `issue_d ≥ 2013` (năm X) — trước 2013, ~22 trường bureau (`tot_cur_bal`, `num_*`, `mo_sin_*`…) bị thiếu hoàn toàn; nếu giữ, "bị thiếu" thực chất là tín hiệu "khoản vay cũ" chỉ có ở train.
4. Chỉ giữ khoản vay **đã kết thúc**: `Fully Paid` → 0; `Charged Off`, `Default` → 1 (gộp các trạng thái "Does not meet the credit policy. Status:…").

**Chọn năm test:** năm phát hành gần nhất mà **≥ 95%** khoản 36 tháng đã kết thúc (tính trước bộ lọc 4). Nếu chọn năm mới hơn, tập test chỉ gồm các khoản kết thúc sớm (vỡ nợ sớm hoặc trả sớm) → lệch.

**Bỏ cột (A5–A7):** ID/văn bản; cột **rò rỉ** chỉ biết sau khi giải ngân (`total_pymnt*`, `recoveries`, `last_pymnt_*`, `last_fico_range_*`, `settlement_*`, `hardship_*`…); `zip_code`; các cột thành hằng số do chính bộ lọc (`application_type`, `term`…).

**Đổi định dạng (A9):** `earliest_cr_line` "Aug-2003" → năm thập phân; `issue_d` **chỉ dùng để chia**, không đưa vào feature.

In [ ]:
LC_FINISHED = {"Fully Paid": 0, "Charged Off": 1, "Default": 1}
LC_POLICY_PREFIX = "Does not meet the credit policy. Status:"
LC_DROP_ID_TEXT = ["id", "member_id", "url", "desc", "title", "emp_title"]
LC_DROP_LEAK = ["loan_status", "recoveries", "collection_recovery_fee", "next_pymnt_d",
                "last_credit_pull_d", "pymnt_plan", "deferral_term", "payment_plan_start_date",
                "orig_projected_additional_accrued_interest"]
LC_DROP_LEAK_PREFIX = ("total_pymnt", "total_rec_", "out_prncp", "last_pymnt_", "last_fico_range_",
                       "debt_settlement_", "settlement_", "hardship_")
LC_DROP_OTHER = ["zip_code"]
LC_DROP_CONSTANT = ["application_type", "verification_status_joint", "term"]  # hằng số do bộ lọc
LC_DATE_TO_YEAR = ["earliest_cr_line", "sec_app_earliest_cr_line"]
LC_CATEGORICAL = ["grade", "sub_grade", "home_ownership", "verification_status", "purpose",
                  "addr_state", "emp_length", "initial_list_status", "disbursement_method"]
LC_INFO = {}   # lưu thông tin trung gian (tỷ lệ đã kết thúc theo năm, năm train/val/test) để EDA


def _skip_bad_row(row):
    return "skip"   # cuối file có vài dòng tổng kết ("Total amount funded...") thiếu cột -> bỏ qua


def _month_to_year(s: pd.Series) -> pd.Series:
    """'Aug-2003' -> 2003 + 7/12 (chỉ đổi định dạng, không tạo thông tin mới)."""
    d = pd.to_datetime(s, format="%b-%Y")
    return d.dt.year + (d.dt.month - 1) / 12


def load_lending_club_b():
    path = RAW / "lending_club" / "accepted_2007_to_2018Q4.csv.gz"
    with pacsv.open_csv(path, read_options=pacsv.ReadOptions(block_size=1 << 20)) as r:
        names = r.schema.names
    # Đọc MỌI cột dưới dạng chuỗi (151 cột thưa làm việc đoán kiểu theo khối bị sai), ép kiểu sau
    reader = pacsv.open_csv(
        path, read_options=pacsv.ReadOptions(block_size=64 << 20),
        convert_options=pacsv.ConvertOptions(column_types={n: pa.string() for n in names},
                                             strings_can_be_null=True),
        parse_options=pacsv.ParseOptions(invalid_row_handler=_skip_bad_row))

    kept, years_all, finished_all, offset = [], [], [], 0
    for batch in reader:
        t = pa.Table.from_batches([batch])
        t = t.filter(pc.is_valid(t["loan_amnt"]))                 # bỏ dòng chân trang
        t = t.append_column("row_id", pa.array(np.arange(offset, offset + t.num_rows, dtype=np.int64)))
        offset += t.num_rows
        # Bộ lọc 1 và 2: hồ sơ cá nhân, kỳ hạn 36 tháng
        t = t.filter(pc.and_(pc.equal(t["application_type"], "Individual"),
                             pc.match_substring(t["term"], "36")))
        status = t["loan_status"].to_pandas().str.replace(LC_POLICY_PREFIX, "", regex=False)
        year = pd.to_datetime(t["issue_d"].to_pandas(), format="%b-%Y").dt.year
        finished = status.isin(list(LC_FINISHED))
        years_all.append(year.to_numpy())                         # dùng để chọn năm test
        finished_all.append(finished.to_numpy())
        keep = ((year >= LC_START_YEAR) & finished).to_numpy()    # bộ lọc 3 và 4
        t = t.filter(pa.array(keep))
        t = t.set_column(t.schema.get_field_index("loan_status"), "loan_status",
                         pa.array(status[keep].to_numpy(), pa.string()))
        kept.append(t)

    # Năm test = năm gần nhất có >= 95% khoản 36 tháng đã kết thúc
    share = pd.Series(np.concatenate(finished_all)).groupby(np.concatenate(years_all)).mean()
    test_year = int(share[share >= 0.95].index.max())
    periods = {"train": (LC_START_YEAR, test_year - 2), "val": (test_year - 1, test_year - 1),
               "test": (test_year, test_year)}
    LC_INFO.update(finished_share_36m=share, periods=periods, n_rows_file=offset)

    table = pa.concat_tables(kept)
    del kept
    issue = pd.to_datetime(table["issue_d"].to_pandas(), format="%b-%Y")
    in_window = ((issue.dt.year >= LC_START_YEAR) & (issue.dt.year <= test_year)).to_numpy()
    table = table.filter(pa.array(in_window))                     # bỏ 2016–2018 (không dùng)

    leak = [c for c in table.column_names if c.startswith(LC_DROP_LEAK_PREFIX)]
    drop = set(LC_DROP_ID_TEXT + LC_DROP_LEAK + leak + LC_DROP_OTHER + LC_DROP_CONSTANT)
    cols = {"row_id": table["row_id"].to_numpy()}
    for c in names:                                               # giữ đúng thứ tự cột gốc
        if c in drop:
            continue
        if c == "issue_d":
            cols[c] = pd.to_datetime(table[c].to_pandas(), format="%b-%Y").to_numpy()
        elif c in LC_DATE_TO_YEAR:
            cols[c] = _month_to_year(table[c].to_pandas()).to_numpy()
        elif c in LC_CATEGORICAL:
            cols[c] = table[c].to_pandas().to_numpy(dtype=object)
        else:                                                     # còn lại phải là số
            cols[c] = pc.cast(table[c], pa.float64()).to_numpy(zero_copy_only=False)
    df = pd.DataFrame(cols)
    df["target"] = table["loan_status"].to_pandas().map(LC_FINISHED).astype("int8").to_numpy()
    return df.reset_index(drop=True)

### 1.5. Bảng khai báo các bộ dữ liệu

Mỗi bộ được mô tả bởi: hàm đọc, danh sách biến phân loại (`None` = mọi cột chuỗi), các cột *không phải feature*, cách chia, và cách điền thiếu ở tầng B.1.

`hmeq_blind` dùng **cùng dữ liệu và cùng fold** với `hmeq`, chỉ khác ở cách điền thiếu (`random`): mọi giá trị thiếu được thay bằng một giá trị rút ngẫu nhiên từ các giá trị quan sát được ở train. Khi đó mô hình **không thể biết ô nào từng bị thiếu** — dùng để đo phần AUC đến từ rò rỉ qua mẫu hình thiếu (mục 12).

In [ ]:
SPECS = {
    "give_me_some_credit": dict(load=load_give_me_some_credit, categorical=[], non_features=[],
                                method="kfold", impute="median"),
    "taiwan_default_credit": dict(load=load_taiwan, categorical=["SEX", "EDUCATION", "MARRIAGE"],
                                  non_features=[], method="kfold", impute="median"),
    "home_credit": dict(load=load_home_credit, categorical=None, non_features=["SK_ID_CURR"],
                        method="kfold", impute="median"),
    "hmeq": dict(load=load_hmeq, categorical=["REASON", "JOB"], non_features=[],
                 method="kfold", impute="median"),
    "hmeq_blind": dict(load=load_hmeq, categorical=["REASON", "JOB"], non_features=[],
                       method="kfold", impute="random"),
    "south_german_credit": dict(
        load=load_south_german_credit, non_features=[], method="kfold", impute="median",
        categorical=["status", "credit_history", "purpose", "savings", "employment_duration",
                     "installment_rate", "personal_status_sex", "other_debtors", "present_residence",
                     "property", "other_installment_plans", "housing", "number_credits", "job",
                     "people_liable", "telephone", "foreign_worker"]),
    "fico_heloc": dict(load=load_fico_heloc, categorical=["MaxDelq2PublicRecLast12M", "MaxDelqEver"],
                       non_features=[], method="kfold", impute="median"),
    "lending_club_B": dict(load=load_lending_club_b, categorical=LC_CATEGORICAL, non_features=["issue_d"],
                           method="time", impute="median"),
}


def declare_columns(name: str, df: pd.DataFrame):
    """A10: tách danh sách feature thành biến số và biến phân loại."""
    spec = SPECS[name]
    features = [c for c in df.columns if c not in {"row_id", "target", *spec["non_features"]}]
    if spec["categorical"] is None:          # Home Credit: mọi cột chuỗi là phân loại
        categorical = [c for c in features if not pd.api.types.is_numeric_dtype(df[c])]
    else:
        categorical = list(spec["categorical"])
    numeric = [c for c in features if c not in categorical]
    # kiểm tra: không được còn cột chuỗi nào bị khai báo nhầm là biến số
    bad = [c for c in numeric if not pd.api.types.is_numeric_dtype(df[c])]
    assert not bad, f"{name}: cột chuỗi chưa khai báo phân loại: {bad}"
    return features, numeric, categorical

### 1.6. Đọc toàn bộ dữ liệu

Bước này mất khoảng 3–5 phút ở chế độ đầy đủ (Lending Club ~2–4 phút, Home Credit ~10 giây).

In [ ]:
DATA, COLS = {}, {}
for name in DATASETS:
    t0 = time.perf_counter()
    if name == "hmeq_blind" and "hmeq" in DATA:
        df = DATA["hmeq"]                     # cùng dữ liệu với hmeq, không cần đọc lại
    else:
        df = SPECS[name]["load"]()
    DATA[name] = df
    COLS[name] = dict(zip(["features", "numeric", "categorical"], declare_columns(name, df)))
    print(f"[{name:22s}] {len(df):>9,} dòng | {len(COLS[name]['numeric']):3d} số + "
          f"{len(COLS[name]['categorical']):2d} phân loại | tỷ lệ xấu {df.target.mean():6.2%} | "
          f"{time.perf_counter() - t0:5.1f}s")

In [ ]:
# Bảng tóm tắt sau tầng A
rows = []
for name, df in DATA.items():
    X = df[COLS[name]["features"]]
    rows.append({
        "Bộ dữ liệu": LABELS[name],
        "Số dòng": len(df),
        "Biến số": len(COLS[name]["numeric"]),
        "Biến phân loại": len(COLS[name]["categorical"]),
        "Tỷ lệ xấu": df.target.mean(),
        "% ô bị thiếu": X.isna().mean().mean(),
        "% dòng có ít nhất 1 ô thiếu": X.isna().any(axis=1).mean(),
        "Dòng trùng lặp feature": int(X.duplicated(keep=False).sum()),
    })
OVERVIEW = pd.DataFrame(rows).set_index("Bộ dữ liệu")
OVERVIEW.to_csv(OUT / "overview.csv")
display(OVERVIEW.style.format({"Số dòng": "{:,}", "Tỷ lệ xấu": "{:.2%}", "% ô bị thiếu": "{:.2%}",
                               "% dòng có ít nhất 1 ô thiếu": "{:.2%}", "Dòng trùng lặp feature": "{:,}"}))

## 2. Khám phá dữ liệu nhanh (EDA)

Mục đích: hiểu **mất cân bằng lớp** (tỷ lệ xấu thấp → accuracy vô nghĩa, cần AUC / precision / recall) và **mức độ thiếu dữ liệu** (quyết định tầm quan trọng của bước điền thiếu). EDA chi tiết từng bộ nằm trong `notebooks/eda_*.ipynb`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
o = OVERVIEW.sort_values("Tỷ lệ xấu")
axes[0].barh(o.index, o["Tỷ lệ xấu"] * 100, color="#C44E52")
axes[0].set_xlabel("Tỷ lệ vỡ nợ (%)")
axes[0].set_title("Mất cân bằng lớp: tỷ lệ nhãn xấu")
for i, v in enumerate(o["Tỷ lệ xấu"] * 100):
    axes[0].text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=8)
o = OVERVIEW.sort_values("% ô bị thiếu")
axes[1].barh(o.index, o["% ô bị thiếu"] * 100, color="#4C72B0", label="% ô bị thiếu")
axes[1].barh(o.index, o["% dòng có ít nhất 1 ô thiếu"] * 100, color="#4C72B0", alpha=0.3,
             label="% dòng có ≥ 1 ô thiếu")
axes[1].set_xlabel("%")
axes[1].set_title("Mức độ thiếu dữ liệu")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "eda_ty_le_xau_va_thieu.png", dpi=130)
plt.show()

**Đọc hình:**
* GMSC (~6,7%), Home Credit (~8%) mất cân bằng mạnh: một mô hình luôn đoán "tốt" đã có accuracy > 90% → không dùng accuracy làm độ đo chính.
* South German (30%), HELOC (~52%) gần cân bằng hơn.
* Home Credit và HMEQ có nhiều ô thiếu → cách điền thiếu ảnh hưởng lớn.

### 2.1. HMEQ: mẫu hình thiếu dữ liệu tiết lộ nhãn

Nếu việc *một ô bị thiếu* tương quan mạnh với nhãn, mô hình có thể học "thiếu → xấu" thay vì học hành vi tín dụng thật. Đó là một dạng **rò rỉ**: trong thực tế, hồ sơ được điền đầy đủ trước khi xét duyệt, tín hiệu này không tồn tại lúc ra quyết định.

In [ ]:
if "hmeq" in DATA:
    h = DATA["hmeq"]
    rows = []
    for c in COLS["hmeq"]["features"]:
        miss = h[c].isna()
        if miss.any():
            rows.append({"Cột": c, "% thiếu": miss.mean(), "Tỷ lệ xấu khi THIẾU": h.target[miss].mean(),
                         "Tỷ lệ xấu khi CÓ": h.target[~miss].mean()})
    t = pd.DataFrame(rows).set_index("Cột").sort_values("Tỷ lệ xấu khi THIẾU", ascending=False)
    display(t.style.format("{:.1%}"))
    n_miss = h[COLS["hmeq"]["features"]].isna().sum(axis=1)
    display(Markdown("**Tỷ lệ xấu theo số ô bị thiếu trong một hồ sơ:**"))
    display(h.groupby(n_miss).target.agg(["size", "mean"])
             .rename(columns={"size": "Số hồ sơ", "mean": "Tỷ lệ xấu"}).T)

Khi `DEBTINC` (tỷ lệ nợ/thu nhập) bị thiếu, tỷ lệ xấu cao gấp nhiều lần so với khi có; hồ sơ thiếu nhiều ô gần như toàn bộ là xấu. Một cây quyết định học được ngay quy tắc này. Phần 12 so sánh `hmeq` với `hmeq_blind` để đo phần AUC "ảo" do rò rỉ.

### 2.2. Lending Club: chọn năm test và tỷ lệ vỡ nợ theo năm

In [ ]:
if "lending_club_B" in DATA:
    share = LC_INFO["finished_share_36m"]
    display(Markdown(f"Số khoản vay trong file gốc: **{LC_INFO['n_rows_file']:,}**. "
                     f"Năm train / val / test: **{LC_INFO['periods']}**"))
    display(share.rename("Tỷ lệ khoản 36 tháng đã kết thúc").to_frame().T.style.format("{:.1%}"))
    lc = DATA["lending_club_B"]
    by_year = lc.groupby(lc.issue_d.dt.year).target.agg(["size", "mean"])
    by_year.columns = ["Số khoản vay", "Tỷ lệ vỡ nợ"]
    display(by_year.T)
    # minh họa lý do chọn năm X = 2013: tỷ lệ thiếu của một vài trường bureau theo năm
    # (tính trên dữ liệu đã lọc nên chỉ còn 2013-2015; xem configs/preprocessing_v1.md cho 2007-2012)
    display(lc.groupby(lc.issue_d.dt.year)[["tot_cur_bal", "mo_sin_old_rev_tl_op", "num_actv_bc_tl"]]
              .apply(lambda g: g.isna().mean()).style.format("{:.2%}"))

* Khoản vay phát hành 2016–2018 phần lớn **chưa kết thúc** tại thời điểm chốt dữ liệu → không dùng làm test.
* Tỷ lệ vỡ nợ tăng dần theo năm (train < val < test): đây là **trôi dạt thật theo thời gian** — một lý do khiến AUC ở Lending Club thấp hơn các bộ chia ngẫu nhiên.

## 3. Chia dữ liệu

### 3.1. Vì sao chia như vậy

| Tầng | Tỷ lệ trên toàn bộ | Dùng cho |
|---|---|---|
| Test (1 fold ngoài) | 20% | **Chỉ để đánh giá cuối cùng** |
| Validation (20% của phần train) | 16% | Chọn ngưỡng quyết định (và tinh chỉnh nếu có) |
| Train thực | 64% | Fit tiền xử lý và mô hình |

* **Stratified**: mỗi fold giữ tỷ lệ xấu gần bằng toàn bộ → AUC giữa các fold so sánh được.
* **Group**: các dòng có **feature giống hệt nhau** (ví dụ GMSC có ~1.000 dòng như vậy) được gom thành một nhóm và luôn nằm cùng phía. Nếu một bản sao ở train và bản kia ở test, mô hình chỉ cần "nhớ" là đoán đúng → AUC bị thổi phồng. Nhóm = mã băm (hash) của toàn bộ giá trị feature.
* **5 fold ngoài**: mỗi dòng nằm trong tập test đúng 1 lần → báo cáo trung bình ± độ lệch chuẩn qua 5 fold.
* **Lending Club**: dữ liệu có thứ tự thời gian, nên chia theo năm phát hành (train 2013, val 2014, test 2015), **không xáo trộn**. Chia ngẫu nhiên sẽ cho mô hình "nhìn thấy tương lai".

In [ ]:
def feature_groups(df: pd.DataFrame, features: list) -> np.ndarray:
    """Mã nhóm = hash của toàn bộ giá trị feature: các dòng giống hệt nhau có cùng mã nhóm."""
    h = pd.util.hash_pandas_object(df[features], index=False).to_numpy()
    return pd.factorize(h)[0]


def group_kfold_roles(df: pd.DataFrame, groups: np.ndarray) -> pd.DataFrame:
    """Trả về bảng row_id -> vai trò (train / val / test) cho từng fold k = 1..5."""
    y = df["target"].to_numpy()
    roles = pd.DataFrame({"row_id": df["row_id"].to_numpy(), "group": groups})
    outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (tr_idx, te_idx) in enumerate(outer.split(np.zeros(len(y)), y, groups), start=1):
        # tách validation = 1 fold của một StratifiedGroupKFold(5) bên trong phần train (~20%)
        inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        fit_rel, val_rel = next(inner.split(np.zeros(len(tr_idx)), y[tr_idx], groups[tr_idx]))
        role = np.empty(len(y), dtype=object)
        role[tr_idx[fit_rel]], role[tr_idx[val_rel]], role[te_idx] = "train", "val", "test"
        roles[f"role_fold{k}"] = role
    return roles


def time_holdout_roles(df: pd.DataFrame, time_col: str, periods: dict) -> pd.DataFrame:
    """Lending Club: một lần chia theo năm; dòng ngoài các khoảng năm = 'unused'."""
    year = df[time_col].dt.year.to_numpy()
    role = np.full(len(df), "unused", dtype=object)
    for part in ("train", "val", "test"):
        lo, hi = periods[part]
        role[(year >= lo) & (year <= hi)] = part
    return pd.DataFrame({"row_id": df["row_id"].to_numpy(), "role_fold1": role})


def n_folds_of(name: str) -> int:
    return sum(c.startswith("role_fold") for c in SPLITS[name].columns)


def check_split(name: str, roles: pd.DataFrame, df: pd.DataFrame):
    """Kiểm tra: không nhóm trùng lặp nào bị chia đôi; test các fold phủ đúng 1 lần mọi dòng;
    Lending Club: train < val < test theo thời gian."""
    tests = []
    for k in range(1, sum(c.startswith("role_fold") for c in roles.columns) + 1):
        r = roles[f"role_fold{k}"]
        tests.append(set(roles.loc[r == "test", "row_id"]))
        if "group" in roles:
            g = {p: set(roles.loc[r == p, "group"]) for p in ("train", "val", "test")}
            assert not (g["train"] & g["test"] or g["val"] & g["test"] or g["train"] & g["val"]), \
                f"{name} fold {k}: một nhóm trùng lặp bị chia ra hai phía"
        else:
            t = df[SPECS[name]["non_features"][0]]
            tr, va, te = (t[(r == p).to_numpy()] for p in ("train", "val", "test"))
            assert tr.max() < va.min() and va.max() < te.min(), "sai thứ tự thời gian"
    if "group" in roles:
        assert sum(map(len, tests)) == len(df) and set().union(*tests) == set(df.row_id)

In [ ]:
SPLITS = {}
for name, df in DATA.items():
    if SPECS[name]["method"] == "time":
        roles = time_holdout_roles(df, "issue_d", LC_INFO["periods"])
    elif name == "hmeq_blind" and "hmeq" in SPLITS:
        roles = SPLITS["hmeq"]                     # cùng fold với hmeq -> so sánh công bằng
    else:
        roles = group_kfold_roles(df, feature_groups(df, COLS[name]["features"]))
    check_split(name, roles, df)
    SPLITS[name] = roles

# Bảng kích thước và tỷ lệ xấu từng fold
rows = []
for name, roles in SPLITS.items():
    y = DATA[name]["target"].to_numpy()
    for k in range(1, n_folds_of(name) + 1):
        r = roles[f"role_fold{k}"].to_numpy()
        row = {"Bộ dữ liệu": LABELS[name], "Fold": k}
        for p in ("train", "val", "test"):
            row[f"n {p}"] = int((r == p).sum())
            row[f"% xấu {p}"] = y[r == p].mean()
        rows.append(row)
FOLD_TABLE = pd.DataFrame(rows)
display(FOLD_TABLE.groupby("Bộ dữ liệu", sort=False).mean(numeric_only=True).drop(columns="Fold")
        .style.format({**{f"n {p}": "{:,.0f}" for p in ("train", "val", "test")},
                       **{f"% xấu {p}": "{:.2%}" for p in ("train", "val", "test")}}))
print("Mọi kiểm tra chia dữ liệu đều đạt (không nhóm trùng lặp nào bị chia đôi).")

Tỷ lệ xấu của train / val / test gần như bằng nhau ở mọi bộ chia ngẫu nhiên — nhờ phân tầng. Riêng Lending Club, tỷ lệ xấu tăng dần theo năm (trôi dạt thời gian, không phải lỗi chia).

## 4. Tầng B.0 — Sàng lọc cột (fit trên train của từng fold)

`SimpleImputer` của scikit-learn **âm thầm bỏ** cột trống hoàn toàn. Bước này làm việc bỏ cột trở nên tường minh, và chỉ nhìn **train** của từng fold:

| Quy tắc | Hành động |
|---|---|
| Cột thiếu 100% ở train | Bỏ |
| Biến số chỉ có 1 giá trị ở train | Bỏ (không mang thông tin) |
| Biến phân loại chỉ có 1 nhóm ở train (NaN tính là một nhóm) | Bỏ |
| Tỷ lệ thiếu giữa train và val/test lệch > 30 điểm % | Giữ, nhưng **cảnh báo** (dấu hiệu dịch chuyển phân phối) |

In [ ]:
def screen_columns(X_train: pd.DataFrame, num_cols: list, cat_cols: list) -> dict:
    """Trả về {cột bị bỏ: lý do}; mọi thống kê chỉ tính trên train."""
    miss = X_train.isna().mean()
    dropped = {c: "thiếu 100% ở train" for c in miss.index[miss == 1.0]}
    for c in num_cols:
        if c not in dropped and X_train[c].nunique(dropna=True) <= 1:
            dropped[c] = "biến số hằng ở train"
    for c in cat_cols:
        if c not in dropped and X_train[c].nunique(dropna=False) <= 1:
            dropped[c] = "biến phân loại hằng ở train"
    return dict(sorted(dropped.items()))


def missing_shift(X_train: pd.DataFrame, X_other: pd.DataFrame, thr: float = MISSING_SHIFT_THR) -> list:
    """Các cột có tỷ lệ thiếu lệch > thr giữa train và một phần khác."""
    diff = X_other.isna().mean() - X_train.isna().mean()
    return sorted(diff.index[diff.abs() > thr])


SCREEN = {}   # SCREEN[name][k] = {"dropped", "numeric_kept", "categorical_kept", "warnings"}
rows = []
for name, df in DATA.items():
    feats, num, cat = COLS[name]["features"], COLS[name]["numeric"], COLS[name]["categorical"]
    SCREEN[name] = {}
    for k in range(1, n_folds_of(name) + 1):
        r = SPLITS[name][f"role_fold{k}"].to_numpy()
        tr = df.loc[r == "train", feats]
        dropped = screen_columns(tr, num, cat)
        kept = [c for c in feats if c not in dropped]
        warn = sorted(set(missing_shift(tr[kept], df.loc[r == "val", kept]))
                      | set(missing_shift(tr[kept], df.loc[r == "test", kept])))
        SCREEN[name][k] = {"dropped": dropped, "warnings": warn,
                           "numeric_kept": [c for c in num if c not in dropped],
                           "categorical_kept": [c for c in cat if c not in dropped]}
        rows.append({"Bộ dữ liệu": LABELS[name], "Fold": k, "Số cột bỏ": len(dropped),
                     "Cột bỏ (ví dụ)": ", ".join(list(dropped)[:6]) + (" …" if len(dropped) > 6 else ""),
                     "Cảnh báo lệch thiếu": len(warn),
                     "Giữ: số + phân loại": f"{len(SCREEN[name][k]['numeric_kept'])} + "
                                             f"{len(SCREEN[name][k]['categorical_kept'])}"})
SCREEN_TABLE = pd.DataFrame(rows)
display(SCREEN_TABLE[SCREEN_TABLE.Fold == 1].drop(columns="Fold").set_index("Bộ dữ liệu"))

* Các bộ nhỏ gần như không mất cột nào.
* Lending Club bỏ ~31 cột: cột đồng vay (`sec_app_*`, `*_joint`) trống vì đã lọc hồ sơ cá nhân; một số trường bureau chỉ có từ 2015–2016 nên trống ở train 2013; `policy_code`, `disbursement_method` là hằng số.
* Không bộ nào có cảnh báo lệch tỷ lệ thiếu > 30 điểm — nhờ bộ lọc năm X = 2013 của Lending Club.

## 5. Tầng B.1 — Tiền xử lý theo Baesens (fit trên train của từng fold)

| Bước | Biến số | Biến phân loại |
|---|---|---|
| 1. Giá trị thiếu | điền **median** của train | thêm nhóm mới `"__missing__"` |
| 2. Mã hóa | — | **one-hot** (classic ML) hoặc **mã số nguyên** (GBDT) |
| 3. Chuẩn hóa | về trung bình 0, phương sai 1 (theo train) | — |

* **Vì sao one-hot cho LR/KNN/NB:** các mô hình này coi đầu vào là số thực; nếu mã hóa `JOB` = 0, 1, 2… mô hình sẽ hiểu sai là có thứ tự và khoảng cách.
* **Vì sao mã số nguyên cho GBDT:** XGBoost, LightGBM, CatBoost có cơ chế xử lý biến phân loại riêng; ta khai báo các cột mã là *categorical* cho thư viện (mục 8).
* **Vì sao chuẩn hóa:** KNN dùng khoảng cách Euclid, LR dùng regularization L2 — cả hai nhạy với thang đo. Cây quyết định không bị ảnh hưởng.
* `handle_unknown="ignore"` / `unknown_value=-1`: nhóm xuất hiện ở test mà chưa từng có ở train → vector 0 (one-hot) hoặc mã −1.
* Không thêm cột đánh dấu thiếu, không cắt ngoại lai, không cân bằng lớp — để giữ đúng giao thức Baesens.

In [ ]:
MISSING = "__missing__"


class RandomDrawImputer(BaseEstimator, TransformerMixin):
    """Điền NaN bằng giá trị rút ngẫu nhiên (có seed) từ các giá trị quan sát được ở train của cột đó.
    Dùng cho hmeq_blind: sau khi điền, không còn dấu vết nào cho biết ô nào từng bị thiếu."""

    def __init__(self, seed: int = SEED):
        self.seed = seed

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.pools_ = [X.iloc[:, j].dropna().to_numpy() for j in range(X.shape[1])]
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        rng = np.random.default_rng(self.seed)
        for j, pool in enumerate(self.pools_):
            miss = X.iloc[:, j].isna().to_numpy()
            if miss.any() and len(pool):
                X.iloc[np.flatnonzero(miss), j] = rng.choice(pool, size=int(miss.sum()))
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def _to_str(X):
    """Ép mã nhóm về chuỗi (ví dụ 1 -> '1') nhưng giữ NaN để bước sau gom vào nhóm '__missing__'."""
    X = pd.DataFrame(X)
    return X.apply(lambda s: s.map(lambda v: v if pd.isna(v) else str(v))).astype(object)


def build_preprocessor(num_cols, cat_cols, family: str, impute: str = "median") -> ColumnTransformer:
    """family = 'classic' (one-hot) hoặc 'gbdt' (mã số nguyên); impute = 'median' hoặc 'random'."""
    num = Pipeline([
        ("impute", SimpleImputer(strategy="median") if impute == "median" else RandomDrawImputer()),
        ("scale", StandardScaler()),
    ])
    encoder = (OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
               if family == "classic" else
               OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.float32))
    cat = Pipeline([
        ("to_str", FunctionTransformer(_to_str, feature_names_out="one-to-one")),
        ("impute", SimpleImputer(strategy="constant", fill_value=MISSING) if impute == "median"
         else RandomDrawImputer()),
        ("encode", encoder),
    ])
    # tên cột đầu ra: num__<cột> và cat__<cột>_<nhóm> (one-hot) hoặc cat__<cột> (mã số)
    return ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)],
                             remainder="drop", verbose_feature_names_out=True)


def fold_matrices(name: str, k: int, family: str, refit: bool = False):
    """Ma trận đã tiền xử lý của fold k: {phần: (X, y, row_id)}, tên cột, tên phần dùng để fit.
    refit=True (Lending Club, quy tắc 3): fit tiền xử lý trên train + val, áp dụng cho test."""
    df = DATA[name]
    role = SPLITS[name][f"role_fold{k}"].to_numpy()
    sc = SCREEN[name][k]
    num, cat = sc["numeric_kept"], sc["categorical_kept"]
    cols = num + cat
    parts = {p: df[role == p] for p in ("train", "val", "test")}
    if refit:
        parts = {"trainval": pd.concat([parts["train"], parts["val"]]), "test": parts["test"]}
    fit_part = "trainval" if refit else "train"
    pre = build_preprocessor(num, cat, family, SPECS[name]["impute"]).fit(parts[fit_part][cols])  # CHỈ train
    out = {p: (pre.transform(d[cols]).astype(np.float32), d["target"].to_numpy().astype(int),
               d["row_id"].to_numpy()) for p, d in parts.items()}
    # kiểm tra: không còn NaN/inf; biến số trên train có trung bình ~ 0
    assert all(np.isfinite(x).all() for x, _, _ in out.values()), "NaN/inf sau tiền xử lý"
    assert np.abs(out[fit_part][0][:, :len(num)].mean(axis=0)).max() < 1e-3 if num else True
    return out, list(pre.get_feature_names_out()), fit_part

**Minh họa trên HMEQ, fold 1:** trước và sau tiền xử lý (nhánh classic và nhánh GBDT).

In [ ]:
demo = "hmeq" if "hmeq" in DATA else DATASETS[0]
role = SPLITS[demo]["role_fold1"].to_numpy()
sc = SCREEN[demo][1]
display(Markdown(f"**Trước** (5 dòng train của `{demo}`):"))
display(DATA[demo].loc[role == "train", sc["numeric_kept"] + sc["categorical_kept"]].head())
for fam in ("classic", "gbdt"):
    m, names_out, _ = fold_matrices(demo, 1, fam)
    display(Markdown(f"**Sau — nhánh `{fam}`**: {m['train'][0].shape[1]} cột "
                     f"(train {m['train'][0].shape[0]:,} dòng, val {m['val'][0].shape[0]:,}, "
                     f"test {m['test'][0].shape[0]:,})"))
    display(pd.DataFrame(m["train"][0][:5], columns=names_out).round(3))

Ở nhánh classic, `JOB` được tách thành nhiều cột 0/1 (`cat__JOB_Mgr`, `cat__JOB___missing__`…); ở nhánh GBDT, `JOB` giữ một cột mã số nguyên. Các biến số có giá trị quanh 0 vì đã chuẩn hóa.

## 6. Độ đo đánh giá

### 6.1. Độ đo không cần ngưỡng (đánh giá điểm xác suất `p`)

| Độ đo | Ý nghĩa | Tốt hơn khi |
|---|---|---|
| **AUC** | xác suất một hồ sơ xấu ngẫu nhiên được chấm điểm cao hơn một hồ sơ tốt ngẫu nhiên; 0,5 = đoán bừa | cao |
| **Gini** | `2·AUC − 1`, cách diễn đạt quen thuộc trong ngành ngân hàng | cao |
| **KS** | khoảng cách lớn nhất giữa phân phối điểm của nhóm xấu và nhóm tốt (`max(TPR − FPR)`) | cao |
| **Brier** | trung bình `(p − y)²`: đo cả phân biệt lẫn **hiệu chỉnh** (xác suất có đúng mức không) | thấp |
| **LogLoss** | `−mean(y·log p + (1−y)·log(1−p))`: phạt nặng dự đoán tự tin mà sai | thấp |
| **AP** | diện tích dưới đường precision–recall; mức cơ sở = tỷ lệ xấu; hữu ích khi lớp xấu hiếm | cao |

**AUC là độ đo chính** để xếp hạng mô hình (như Baesens et al. 2026): không phụ thuộc ngưỡng, không bị ảnh hưởng bởi mất cân bằng lớp.

### 6.2. Độ đo theo ngưỡng

Để ra quyết định duyệt / từ chối cần một **ngưỡng** `t`: `p ≥ t` → dự đoán "xấu". Ngưỡng được chọn **trên validation** sao cho F1 lớn nhất, rồi **giữ nguyên** khi đánh giá trên test (chọn ngưỡng trên test là gian lận).

* **Precision** = trong các hồ sơ bị đoán xấu, bao nhiêu % thật sự xấu (chi phí từ chối nhầm khách tốt).
* **Recall** = trong các hồ sơ thật sự xấu, bắt được bao nhiêu % (chi phí cho vay nhầm khách xấu).
* **F1** = trung bình điều hòa của precision và recall.

In [ ]:
def ks_stat(y, p) -> float:
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))


def f1_threshold(y_val, p_val) -> float:
    """Ngưỡng trên validation làm F1 lớn nhất."""
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[int(np.argmax(f1))])


def score(y, p, threshold=None) -> dict:
    """Tính toàn bộ độ đo cho nhãn y và xác suất dự đoán p (và ngưỡng nếu có)."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    auc = roc_auc_score(y, p)
    out = {"auc": auc, "gini": 2 * auc - 1, "ks": ks_stat(y, p),
           "brier": brier_score_loss(y, p),
           "logloss": log_loss(y, np.clip(p, 1e-15, 1 - 1e-15)),   # cắt để tránh log(0)
           "ap": average_precision_score(y, p),
           "bad_rate": y.mean(), "mean_pred": p.mean()}
    if threshold is not None:
        yhat = (p >= threshold).astype(int)
        out.update(threshold=threshold,
                   precision=precision_score(y, yhat, zero_division=0),
                   recall=recall_score(y, yhat, zero_division=0),
                   f1=f1_score(y, yhat, zero_division=0),
                   accuracy=accuracy_score(y, yhat))
    return {k: float(v) for k, v in out.items()}


def bootstrap_auc(y, p, n=N_BOOTSTRAP, seed=SEED) -> dict:
    """Khoảng tin cậy 95% (phân vị) của AUC test bằng bootstrap — dùng cho Lending Club (chỉ 1 lần chia)."""
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))      # rút có hoàn lại cùng số dòng
        if y[idx].min() != y[idx].max():
            aucs.append(roc_auc_score(y[idx], p[idx]))
    aucs = np.array(aucs)
    return {"mean": aucs.mean(), "std": aucs.std(ddof=1),
            "ci95": (np.percentile(aucs, 2.5), np.percentile(aucs, 97.5))}


# Ví dụ nhỏ để kiểm tra trực giác
_y = np.array([0, 0, 0, 1, 1])
_p = np.array([0.1, 0.4, 0.35, 0.8, 0.3])
print({k: round(v, 3) for k, v in score(_y, _p, threshold=0.35).items()})

## 7. Mô hình và huấn luyện — nhóm Classic ML

Tất cả mô hình dùng **tham số mặc định** của thư viện (chỉ đặt seed, số luồng, số vòng lặp tối đa của LR). Như vậy so sánh phản ánh "mô hình nào tốt sẵn", chưa tính tinh chỉnh.

| Mô hình | Ý tưởng | Tham số chính (mặc định) | Đầu vào |
|---|---|---|---|
| **LR** — Hồi quy logistic | `p = σ(w·x + b)`; tuyến tính trên log-odds, dễ giải thích; **chuẩn ngành** cho chấm điểm tín dụng | C = 1, L2, lbfgs, `max_iter=2000` | one-hot + chuẩn hóa |
| **KNN** — k láng giềng | xác suất xấu = tỷ lệ xấu trong 5 hồ sơ train gần nhất (Euclid) | k = 5, trọng số đều | one-hot + chuẩn hóa |
| **RF** — Rừng ngẫu nhiên | trung bình 100 cây quyết định, mỗi cây học trên mẫu bootstrap và tập con feature | 100 cây, cây mọc hết, `sqrt` feature | one-hot + chuẩn hóa |
| **NB** — Gaussian Naive Bayes | giả định các feature độc lập và phân phối chuẩn theo từng lớp | — | one-hot + chuẩn hóa |

**Quy trình cho mỗi bộ dữ liệu × fold × mô hình:**
1. Fit mô hình trên **train**.
2. Dự đoán xác suất trên train, val, test (KNN không chấm train: mỗi điểm train là láng giềng gần nhất của chính nó → AUC train vô nghĩa).
3. Chọn ngưỡng F1 trên **val**; tính mọi độ đo trên từng phần.
4. **Lending Club** (quy tắc 3): fit lại tiền xử lý + mô hình trên **train + val** (2013–2014), đánh giá trên test 2015; ngưỡng lấy từ mô hình chỉ-train ở bước 3; thêm khoảng tin cậy bootstrap.

Mỗi bộ dữ liệu chỉ tiền xử lý **một lần mỗi fold** rồi dùng chung cho cả 4 mô hình (cùng ma trận → so sánh công bằng).

In [ ]:
# Hàm tạo mô hình classic (mỗi lần gọi trả về một mô hình mới, chưa fit)
FACTORIES = {
    "LR": lambda: LogisticRegression(max_iter=2000, random_state=SEED),
    "KNN": lambda: KNeighborsClassifier(n_jobs=-1),
    "RF": lambda: RandomForestClassifier(n_jobs=-1, random_state=SEED),
    "NB": lambda: GaussianNB(),
}
NATIVE = set(GBDT_MODELS)        # GBDT: khai báo cột mã là categorical cho thư viện
SKIP_TRAIN_SCORE = {"KNN"}
CAT_PREFIX = "cat__"

# Nơi lưu kết quả: khóa (model, dataset) -> ... ; chạy lại một ô sẽ ghi đè, không bị nhân đôi
FOLD_ROWS = {}    # danh sách dòng độ đo (dataset, model, fold, part, fit, auc, ...)
PREDS = {}        # DataFrame dự đoán: row_id, fold, part, fit, target, proba
FIT_INFO = {}     # thời gian fit / predict, hội tụ
BOOT = {}         # bootstrap AUC của Lending Club


def model_inputs(model_name, mats, feat_names, fit_part):
    """Chuyển ma trận float32 thành đầu vào phù hợp với từng thư viện.
    - classic: numpy array
    - XGB / LGBM: DataFrame, cột cat__ có kiểu pandas 'category' với các nhóm đã thấy ở train
      (mã -1 = nhóm chưa thấy -> thành giá trị thiếu)
    - CatBoost: DataFrame, cột cat__ là số nguyên, khai báo qua cat_features (-1 = thêm một nhóm)"""
    if model_name not in NATIVE:
        return {p: x for p, (x, _, _) in mats.items()}
    frames = {p: pd.DataFrame(x, columns=feat_names) for p, (x, _, _) in mats.items()}
    cats = [c for c in feat_names if c.startswith(CAT_PREFIX)]
    levels = {c: range(int(frames[fit_part][c].max()) + 1) for c in cats}
    for p, f in frames.items():
        for c in cats:
            codes = f[c].astype(np.int32)
            f[c] = codes if model_name == "CatBoost" else pd.Categorical(codes, categories=levels[c])
    return frames


def fit_model(model_name, X, y):
    model = FACTORIES[model_name]()
    kw = {}
    if model_name == "CatBoost":
        kw["cat_features"] = [c for c in X.columns if c.startswith(CAT_PREFIX)]
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y, **kw)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    return model, {"fit_seconds": time.perf_counter() - t0, "converged": converged}


def run_family(family: str, model_names: list, datasets: list):
    """Huấn luyện và đánh giá một nhóm mô hình trên mọi bộ dữ liệu và mọi fold."""
    for name in datasets:
        t_ds = time.perf_counter()
        for m in model_names:
            FOLD_ROWS[(m, name)], PREDS[(m, name)], FIT_INFO[(m, name)] = [], [], []
        thresholds = {m: {} for m in model_names}
        for k in range(1, n_folds_of(name) + 1):
            mats, feat_names, fit_part = fold_matrices(name, k, family)      # tiền xử lý 1 lần / fold
            for m in model_names:
                X = model_inputs(m, mats, feat_names, fit_part)
                model, info = fit_model(m, X["train"], mats["train"][1])
                parts = ("val", "test") if m in SKIP_TRAIN_SCORE else ("train", "val", "test")
                t0 = time.perf_counter()
                p = {part: model.predict_proba(X[part])[:, 1] for part in parts}
                info["predict_seconds"] = time.perf_counter() - t0
                thr = f1_threshold(mats["val"][1], p["val"])                 # ngưỡng chọn trên VAL
                thresholds[m][k] = thr
                for part in parts:
                    FOLD_ROWS[(m, name)].append({"dataset": name, "model": m, "fold": k, "part": part,
                                                 "fit": "train", **score(mats[part][1], p[part], thr)})
                for part in ("val", "test"):
                    PREDS[(m, name)].append(pd.DataFrame({
                        "row_id": mats[part][2], "fold": k, "part": part, "fit": "train",
                        "target": mats[part][1], "proba": p[part]}))
                FIT_INFO[(m, name)].append({"dataset": name, "model": m, "where": f"fold{k}", **info})
            del mats

        if SPECS[name]["method"] == "time":          # Lending Club: fit lại trên train + val
            mats, feat_names, fit_part = fold_matrices(name, 1, family, refit=True)
            for m in model_names:
                X = model_inputs(m, mats, feat_names, fit_part)
                model, info = fit_model(m, X["trainval"], mats["trainval"][1])
                t0 = time.perf_counter()
                p_test = model.predict_proba(X["test"])[:, 1]
                info["predict_seconds"] = time.perf_counter() - t0
                y_test = mats["test"][1]
                FOLD_ROWS[(m, name)].append({"dataset": name, "model": m, "fold": 1, "part": "test",
                                             "fit": "train+val", **score(y_test, p_test, thresholds[m][1])})
                PREDS[(m, name)].append(pd.DataFrame({
                    "row_id": mats["test"][2], "fold": 1, "part": "test", "fit": "train+val",
                    "target": y_test, "proba": p_test}))
                FIT_INFO[(m, name)].append({"dataset": name, "model": m, "where": "refit", **info})
                BOOT[(m, name)] = bootstrap_auc(y_test, p_test)
            del mats

        # in tóm tắt và lưu ngay (chạy dài -> không mất kết quả nếu kernel dừng giữa chừng)
        for m in model_names:
            rows = pd.DataFrame(FOLD_ROWS[(m, name)])
            test = rows[rows.part == "test"]
            final = test[test.fit == "train+val"] if (test.fit == "train+val").any() else test
            extra = (f" (bootstrap KTC 95% {BOOT[(m, name)]['ci95'][0]:.4f}–{BOOT[(m, name)]['ci95'][1]:.4f})"
                     if (m, name) in BOOT else f" ± {final.auc.std(ddof=1):.4f}")
            fit_s = sum(i["fit_seconds"] + i["predict_seconds"] for i in FIT_INFO[(m, name)])
            print(f"  [{LABELS[name]:16s}] {m:8s} AUC test {final.auc.mean():.4f}{extra} | {fit_s:6.1f}s")
            pd.concat(PREDS[(m, name)], ignore_index=True).to_parquet(
                OUT / "predictions" / f"{m}__{name}.parquet", index=False)
        save_results()
        print(f"  -> {LABELS[name]} xong trong {time.perf_counter() - t_ds:.0f}s")


def save_results():
    pd.DataFrame([r for v in FOLD_ROWS.values() for r in v]).to_csv(OUT / "fold_metrics.csv", index=False)
    pd.DataFrame([r for v in FIT_INFO.values() for r in v]).to_csv(OUT / "fit_info.csv", index=False)
    pd.DataFrame([{"model": m, "dataset": d, "mean": b["mean"], "std": b["std"],
                   "ci_low": b["ci95"][0], "ci_high": b["ci95"][1]} for (m, d), b in BOOT.items()]
                 ).to_csv(OUT / "bootstrap_lc.csv", index=False)

In [ ]:
# ===== HUẤN LUYỆN NHÓM CLASSIC: LR, KNN, RF, NB =====
# Thời gian ước tính (chế độ đầy đủ): ~10 phút (KNN trên Home Credit và Lending Club chậm nhất)
if not DUNG_KET_QUA_CU:
    t0 = time.perf_counter()
    run_family("classic", CLASSIC_MODELS, DATASETS)
    print(f"\nNhóm classic xong trong {(time.perf_counter() - t0) / 60:.1f} phút")
else:
    print("Bỏ qua — dùng kết quả đã lưu")

## 8. Huấn luyện nhóm GBDT (Gradient Boosted Decision Trees)

**Boosting**: xây cây quyết định *tuần tự*, mỗi cây mới học sửa phần sai (gradient của hàm mất mát) của tổng các cây trước. Ba thư viện khác nhau ở cách mọc cây và cách xử lý biến phân loại:

| Mô hình | Mặc định | Đặc điểm | Biến phân loại |
|---|---|---|---|
| **XGBoost** | 100 cây, sâu 6, learning rate 0,3, `hist` | mọc cây theo tầng, regularization L1/L2 | `enable_categorical=True`: chia nhánh theo tập nhóm |
| **LightGBM** | 100 cây, 31 lá, learning rate 0,1 | mọc cây theo lá (leaf-wise), rất nhanh | cột kiểu `category` của pandas |
| **CatBoost** | 1.000 cây, sâu 6, learning rate tự chọn | cây đối xứng, *ordered boosting* chống overfit | target statistics có thứ tự (`cat_features`) |

Cả ba **tự xử lý giá trị thiếu**, nhưng để giữ đúng giao thức, chúng nhận cùng ma trận đã điền thiếu như classic (nhánh `gbdt` của tầng B.1). Bây giờ mới import thư viện GBDT (xem lưu ý OpenMP ở mục 0).

In [ ]:
import xgboost
import lightgbm
import catboost

print("xgboost", xgboost.__version__, "| lightgbm", lightgbm.__version__, "| catboost", catboost.__version__)

FACTORIES.update({
    "XGB": lambda: xgboost.XGBClassifier(random_state=SEED, n_jobs=-1, enable_categorical=True),
    "LGBM": lambda: lightgbm.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1),
    "CatBoost": lambda: catboost.CatBoostClassifier(random_seed=SEED, thread_count=-1, verbose=0,
                                                    allow_writing_files=False),
})

In [ ]:
# ===== HUẤN LUYỆN NHÓM GBDT: XGB, LGBM, CatBoost =====
# Thời gian ước tính (chế độ đầy đủ): ~15–20 phút (CatBoost 1.000 cây trên Home Credit chậm nhất)
if not DUNG_KET_QUA_CU:
    t0 = time.perf_counter()
    run_family("gbdt", GBDT_MODELS, DATASETS)
    print(f"\nNhóm GBDT xong trong {(time.perf_counter() - t0) / 60:.1f} phút")
else:
    print("Bỏ qua — dùng kết quả đã lưu")

## 9. Tổng hợp kết quả

Từ đây trở đi mọi phân tích đọc lại kết quả đã lưu ở `reports/notebook_full/` (nên có thể chạy riêng phần này với `NB_REUSE=1`).

**Con số báo cáo cho mỗi bộ × mô hình** = trung bình (± độ lệch chuẩn) của độ đo trên **test** qua 5 fold; Lending Club = mô hình fit lại trên train + val, đánh giá trên test 2015 (một con số, độ bất định đo bằng bootstrap).

In [ ]:
FM = pd.read_csv(OUT / "fold_metrics.csv")
FIT = pd.read_csv(OUT / "fit_info.csv")
BOOT_DF = pd.read_csv(OUT / "bootstrap_lc.csv") if (OUT / "bootstrap_lc.csv").stat().st_size > 1 else pd.DataFrame()
METRICS = ["auc", "gini", "ks", "brier", "logloss", "ap", "precision", "recall", "f1", "accuracy"]


def load_preds(model, dataset, part="test"):
    """Dự đoán cuối cùng trên test: 5 fold gộp lại; Lending Club: mô hình fit lại train + val."""
    pr = pd.read_parquet(OUT / "predictions" / f"{model}__{dataset}.parquet").query("part == @part")
    return pr[pr.fit == "train+val"] if (pr.fit == "train+val").any() else pr


def summarise(fm: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (name, m), g in fm.groupby(["dataset", "model"], sort=False):
        test = g[g.part == "test"]
        final = test[test.fit == "train+val"] if (test.fit == "train+val").any() else test
        row = {"dataset": name, "model": m, "n": len(final)}
        for met in METRICS:
            row[met] = final[met].mean()
            row[f"{met}_std"] = final[met].std(ddof=1) if len(final) > 1 else np.nan
        row["threshold"] = test[test.fit == "train"].threshold.mean()
        row["bad_rate"] = final.bad_rate.mean()
        row["train_auc"] = g[g.part == "train"].auc.mean() if (g.part == "train").any() else np.nan
        row["val_auc"] = g[g.part == "val"].auc.mean()
        row["test_auc_fit_train"] = test[test.fit == "train"].auc.mean()
        out.append(row)
    return pd.DataFrame(out)


SUMMARY = summarise(FM)
SUMMARY.to_csv(OUT / "summary.csv", index=False)
models_ok = [m for m in MODELS if m in set(SUMMARY.model)]
ds_ok = [d for d in DATASETS if d in set(SUMMARY.dataset)]
main_ok = [d for d in MAIN_DATASETS if d in ds_ok]


def pivot(metric, datasets=None):
    t = SUMMARY.pivot(index="dataset", columns="model", values=metric)
    return t.loc[datasets or ds_ok, models_ok]


def fmt_pm(metric, datasets=None):
    """Bảng 'trung bình ± độ lệch chuẩn', in đậm giá trị tốt nhất mỗi dòng."""
    m, s = pivot(metric, datasets), pivot(f"{metric}_std", datasets)
    higher = metric not in ("brier", "logloss")
    best = m.idxmax(axis=1) if higher else m.idxmin(axis=1)
    t = m.copy().astype(object)
    for d in m.index:
        for c in m.columns:
            v = f"{m.loc[d, c]:.4f}" + ("" if np.isnan(s.loc[d, c]) else f" ± {s.loc[d, c]:.4f}")
            t.loc[d, c] = f"**{v}**" if c == best[d] else v
    t.index = [LABELS[d] for d in t.index]
    t["Tốt nhất"] = best.values
    return t


display(Markdown("### Bảng 1 — AUC test (trung bình ± độ lệch chuẩn qua 5 fold; Lending Club: fit lại train + val)"))
display(Markdown(fmt_pm("auc").to_markdown()))

In [ ]:
# Bản đồ nhiệt AUC: màu theo AUC, số trong ô; dễ thấy mô hình nào mạnh / yếu ở bộ nào
auc = pivot("auc")
fig, ax = plt.subplots(figsize=(9, 0.55 * len(auc) + 1.5))
im = ax.imshow(auc.values, cmap="RdYlGn", vmin=0.5, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(auc.columns)), auc.columns)
ax.set_yticks(range(len(auc)), [LABELS[d] for d in auc.index])
ax.grid(False)
for i in range(auc.shape[0]):
    best = np.nanargmax(auc.values[i])
    for j in range(auc.shape[1]):
        ax.text(j, i, f"{auc.values[i, j]:.3f}", ha="center", va="center", fontsize=9,
                fontweight="bold" if j == best else "normal")
fig.colorbar(im, ax=ax, label="AUC test")
ax.set_title("AUC test theo bộ dữ liệu × mô hình (in đậm: tốt nhất)")
fig.tight_layout()
fig.savefig(FIG / "auc_heatmap.png", dpi=130)
plt.show()

In [ ]:
# Biểu đồ cột AUC có thanh sai số (độ lệch chuẩn qua 5 fold)
fig, ax = plt.subplots(figsize=(13, 4.4))
w = 0.8 / len(models_ok)
x = np.arange(len(ds_ok))
for i, m in enumerate(models_ok):
    ax.bar(x + (i - (len(models_ok) - 1) / 2) * w, pivot("auc")[m], w,
           yerr=pivot("auc_std")[m].fillna(0), capsize=2, label=m, color=COLORS[m])
ax.set_xticks(x, [LABELS[d] for d in ds_ok], fontsize=9)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("AUC test")
ax.set_title("So sánh 7 mô hình (tham số mặc định)")
ax.legend(ncol=len(models_ok), fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "auc_by_model.png", dpi=130)
plt.show()

In [ ]:
# Các độ đo không cần ngưỡng khác
for met, title in [("gini", "Gini"), ("ks", "KS"), ("ap", "Average Precision (AP)"),
                   ("brier", "Brier (thấp hơn = tốt hơn)"), ("logloss", "LogLoss (thấp hơn = tốt hơn)")]:
    display(Markdown(f"**{title}**"))
    display(Markdown(fmt_pm(met).to_markdown()))

**Lưu ý khi đọc Brier / LogLoss:** hai độ đo này phạt cả **hiệu chỉnh kém**. Naive Bayes thường cho xác suất rất cực đoan (gần 0 hoặc 1) vì giả định độc lập bị vi phạm → LogLoss rất lớn, dù AUC có thể vẫn chấp nhận được.

## 10. Kiểm định thống kê — khác biệt có ý nghĩa hay chỉ do ngẫu nhiên?

Chênh AUC 0,005 giữa hai mô hình có thể chỉ là nhiễu. Theo **Demšar (2006)**, khi so sánh nhiều mô hình trên nhiều bộ dữ liệu:

1. **Xếp hạng** các mô hình trong từng bộ (1 = tốt nhất), lấy **hạng trung bình** qua các bộ. Dùng hạng thay cho AUC vì AUC ở các bộ khác nhau không cùng thang (HMEQ ~0,95, Lending Club ~0,70).
2. **Kiểm định Friedman** (hiệu chỉnh Iman–Davenport): giả thuyết H₀ "mọi mô hình như nhau". p < 0,05 → bác bỏ, có ít nhất một khác biệt.
3. **Post-hoc**:
   * **Nemenyi**: hai mô hình khác biệt có ý nghĩa nếu hạng trung bình cách nhau hơn **CD** (critical difference) `= q_α · √(k(k+1)/(6N))`.
   * So sánh từng mô hình với mô hình hạng tốt nhất (kiểm định z trên hạng, hiệu chỉnh **Holm**).
   * **Wilcoxon** từng cặp (signed-rank trên chênh AUC qua các bộ), hiệu chỉnh Holm cho nhiều phép so sánh.

**Hạn chế:** chỉ có N = 7 bộ dữ liệu → kiểm định có lực yếu; "không có ý nghĩa" **không** có nghĩa là "bằng nhau".

In [ ]:
HIGHER_IS_BETTER = {"auc": True, "gini": True, "ks": True, "ap": True, "f1": True, "precision": True,
                    "recall": True, "brier": False, "logloss": False}


def ranks(metric, datasets):
    return pivot(metric, datasets).rank(axis=1, ascending=not HIGHER_IS_BETTER[metric], method="average")


def holm(pvals):
    """Hiệu chỉnh Holm–Bonferroni: p đã sắp xếp tăng dần nhân với (m − i), giữ tính đơn điệu."""
    order = np.argsort(pvals)
    adj, running, m = np.empty(len(pvals)), 0.0, len(pvals)
    for i, j in enumerate(order):
        running = max(running, min(1.0, (m - i) * pvals[j]))
        adj[j] = running
    return adj


R_AUC = ranks("auc", main_ok)
display(Markdown(f"### Bảng 2 — Hạng theo AUC trên {len(main_ok)} bộ chính (1 = tốt nhất)"))
t = R_AUC.copy()
t.index = [LABELS[d] for d in t.index]
t.loc["**Hạng trung bình**"] = R_AUC.mean()
display(t.round(2))

display(Markdown("### Bảng 3 — Hạng trung bình theo từng độ đo"))
display(pd.DataFrame({met.upper(): ranks(met, main_ok).mean() for met in HIGHER_IS_BETTER}).T.round(2))

In [ ]:
def friedman(table: pd.DataFrame) -> dict:
    """table: bộ dữ liệu × mô hình (giá trị cao = tốt). Friedman chi-square + Iman–Davenport F."""
    n, k = table.shape
    chi2, p = stats.friedmanchisquare(*[table[c] for c in table.columns])
    ff = (n - 1) * chi2 / (n * (k - 1) - chi2) if n * (k - 1) != chi2 else np.inf
    p_ff = 1 - stats.f.cdf(ff, k - 1, (k - 1) * (n - 1))
    return {"N (bộ dữ liệu)": n, "k (mô hình)": k, "chi2_F": chi2, "p (chi2)": p,
            "F Iman–Davenport": ff, "p (F)": p_ff}


FR = friedman(pivot("auc", main_ok))
display(Markdown("### Bảng 4 — Kiểm định Friedman trên AUC"))
display(pd.Series(FR).to_frame("giá trị").T)
print("=> Bác bỏ H0 (có khác biệt giữa các mô hình)" if FR["p (F)"] < 0.05
      else "=> Chưa đủ bằng chứng bác bỏ H0")

In [ ]:
# Giá trị q_0.05 / sqrt(2) của phân phối studentized range cho Nemenyi (Demšar 2006, bảng 5)
NEMENYI_Q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164}

mean_ranks = R_AUC.mean()
k, n = len(mean_ranks), len(main_ok)
se = np.sqrt(k * (k + 1) / (6 * n))
CD = NEMENYI_Q05[k] * se
control = mean_ranks.idxmin()                 # mô hình hạng tốt nhất
rows = []
for m, r in mean_ranks.items():
    if m == control:
        continue
    z = (r - mean_ranks[control]) / se
    rows.append({"Mô hình": m, f"Chênh hạng so với {control}": r - mean_ranks[control], "z": z,
                 "p": 2 * (1 - stats.norm.cdf(abs(z)))})
VS = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
VS["p (Holm)"] = holm(VS.p.to_numpy())
VS["Khác biệt có ý nghĩa (α=0,05)"] = np.where(VS["p (Holm)"] < 0.05, "có", "không")
display(Markdown(f"### Bảng 5 — So với mô hình hạng tốt nhất ({control}); Nemenyi CD = {CD:.3f}"))
display(VS.round(4))

In [ ]:
def cd_diagram(mean_ranks: pd.Series, cd: float, path):
    """Biểu đồ critical difference: trục là hạng trung bình; thanh đỏ dài đúng CD.
    Hai mô hình cách nhau ít hơn CD: chưa khác biệt có ý nghĩa."""
    mr = mean_ranks.sort_values()
    k = len(mr)
    fig, ax = plt.subplots(figsize=(9, 0.45 * k + 1.6))
    ax.set_xlim(1, k)
    ax.set_ylim(0, k + 1.5)
    ax.invert_xaxis()
    ax.axhline(k + 0.8, color="k", lw=1)
    ax.set_yticks([])
    ax.grid(False)
    for i, (m, r) in enumerate(mr.items()):
        y = k - i
        ax.plot([r, r], [y, k + 0.8], color=COLORS.get(m, "k"), lw=1)
        ax.plot(r, k + 0.8, "o", color=COLORS.get(m, "k"))
        ax.text(r, y - 0.1, f"{m} ({r:.2f})", ha="center", va="top", fontsize=9)
    ax.plot([mr.iloc[0], mr.iloc[0] + cd], [k + 1.25] * 2, color="red", lw=3)
    ax.text(mr.iloc[0] + cd / 2, k + 1.35, f"CD = {cd:.2f}", ha="center", va="bottom", color="red", fontsize=9)
    ax.set_xlabel("Hạng trung bình theo AUC (1 = tốt nhất). Cách nhau < CD: chưa khác biệt có ý nghĩa "
                  "(Nemenyi, α = 0,05)")
    for sp in ("left", "right", "top"):
        ax.spines[sp].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.show()


cd_diagram(mean_ranks, CD, FIG / "cd_diagram.png")

In [ ]:
# Wilcoxon signed-rank từng cặp trên AUC của các bộ dữ liệu, hiệu chỉnh Holm
auc_main = pivot("auc", main_ok)
rows = []
for a, b in itertools.combinations(auc_main.columns, 2):
    d = auc_main[a] - auc_main[b]
    stat, p = stats.wilcoxon(auc_main[a], auc_main[b], zero_method="wilcox", alternative="two-sided")
    rows.append({"Cặp": f"{a} vs {b}", "A thắng / B thắng": f"{int((d > 0).sum())} / {int((d < 0).sum())}",
                 "Chênh AUC TB (A − B)": d.mean(), "W": stat, "p": p})
WX = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
WX["p (Holm)"] = holm(WX.p.to_numpy())
display(Markdown("### Bảng 6 — Wilcoxon từng cặp (AUC, hiệu chỉnh Holm)"))
display(WX.round(4))

**Với N = 7**, giá trị p nhỏ nhất mà Wilcoxon hai phía có thể đạt là 2/2⁷ ≈ 0,016 (khi một mô hình thắng ở cả 7 bộ). Sau hiệu chỉnh Holm cho 21 cặp, gần như không cặp nào đạt ngưỡng 0,05 — đây là giới hạn của số bộ dữ liệu, không phải bằng chứng hai mô hình bằng nhau. Vì vậy bổ sung thêm so sánh **trong từng bộ**:

In [ ]:
# So sánh trong từng bộ: số fold mỗi mô hình thắng LR (AUC test, mô hình fit trên train)
t = FM[(FM.part == "test") & (FM.fit == "train") & FM.dataset.isin(ds_ok)].pivot_table(
    index=["dataset", "fold"], columns="model", values="auc")
WINS = pd.DataFrame({d: {m: f"{int((g[m] > g['LR']).sum())}/{len(g)}" for m in models_ok if m != "LR"}
                     for d, g in t.groupby(level=0, sort=False)}).T.loc[ds_ok]
WINS.index = [LABELS[d] for d in WINS.index]
display(Markdown("### Bảng 7 — Số fold thắng LR (AUC test)"))
display(WINS)

In [ ]:
# Lending Club: chỉ có 1 lần chia nên dùng bootstrap GHÉP CẶP trên cùng tập test:
# mỗi lần lặp rút lại các dòng test, tính AUC của mô hình và của LR trên CÙNG mẫu đó, lấy hiệu.
def weighted_auc_factory(y, p):
    """f(w) = AUC với trọng số dòng w (số lần dòng được rút); hòa điểm tính 1/2. Nhanh hơn roc_auc_score
    nhiều lần vì nhóm điểm số một lần rồi chỉ cộng dồn."""
    uniq, inv = np.unique(p, return_inverse=True)
    pos, neg = y.astype(float), 1.0 - y

    def auc(w):
        pw = np.bincount(inv, weights=w * pos, minlength=len(uniq))
        nw = np.bincount(inv, weights=w * neg, minlength=len(uniq))
        below = np.cumsum(nw) - nw
        return float((pw * (below + 0.5 * nw)).sum() / (pw.sum() * nw.sum()))
    return auc


if "lending_club_B" in ds_ok:
    preds = {m: load_preds(m, "lending_club_B").set_index("row_id").sort_index() for m in models_ok}
    y = preds["LR"]["target"].to_numpy()
    rng = np.random.default_rng(SEED)
    weights = [np.bincount(rng.integers(0, len(y), len(y)), minlength=len(y)).astype(float)
               for _ in range(N_BOOTSTRAP)]
    f = {m: weighted_auc_factory(y, preds[m]["proba"].to_numpy()) for m in models_ok}
    ref = np.array([f["LR"](w) for w in weights])
    rows = []
    for m in models_ok:
        if m == "LR":
            continue
        diffs = np.array([f[m](w) for w in weights]) - ref
        one = np.ones(len(y))
        rows.append({"Mô hình": f"{m} − LR", "Chênh AUC": f[m](one) - f["LR"](one),
                     "KTC 95% thấp": np.percentile(diffs, 2.5), "KTC 95% cao": np.percentile(diffs, 97.5),
                     "p hai phía": min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))})
    display(Markdown(f"### Bảng 8 — Lending Club: chênh AUC so với LR (bootstrap ghép cặp, {N_BOOTSTRAP} lần, "
                     f"{len(y):,} dòng test)"))
    display(pd.DataFrame(rows).round(4))
    display(Markdown("**Khoảng tin cậy 95% của AUC test từng mô hình (bootstrap):**"))
    display(BOOT_DF.set_index("model")[["mean", "std", "ci_low", "ci_high"]].loc[models_ok].round(4))
else:
    print("Lending Club không có trong lần chạy này")

Khoảng tin cậy **không chứa 0** → chênh lệch so với LR có ý nghĩa thống kê trên tập test Lending Club. Với ~280 nghìn dòng test, ngay cả chênh 0,002 AUC cũng có thể có ý nghĩa thống kê — nhưng chưa chắc có ý nghĩa thực tế.

## 11. Đường ROC, đường Precision–Recall và độ đo theo ngưỡng

* **ROC**: trục hoành = tỷ lệ khách tốt bị từ chối nhầm (FPR), trục tung = tỷ lệ khách xấu bị bắt (TPR) khi quét mọi ngưỡng. Đường càng sát góc trên trái càng tốt; đường chéo = đoán bừa.
* **Precision–Recall**: hữu ích hơn khi lớp xấu hiếm; đường nằm ngang ở mức tỷ lệ xấu = đoán bừa. Chấm tròn = **điểm vận hành** tại ngưỡng F1 chọn trên validation.

Dự đoán test của 5 fold được gộp lại (mỗi dòng xuất hiện đúng một lần).

In [ ]:
def grid(n, ncols=4, w=4.2, h=3.5):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False)
    for ax in axes.flat[n:]:
        ax.axis("off")
    return fig, axes.flat


# --- Đường ROC ---
fig, axes = grid(len(ds_ok))
for ax, d in zip(axes, ds_ok):
    for m in models_ok:
        pr = load_preds(m, d)
        fpr, tpr, _ = roc_curve(pr.target, pr.proba)
        ax.plot(fpr, tpr, lw=1.1, color=COLORS[m], label=f"{m} ({roc_auc_score(pr.target, pr.proba):.3f})")
    ax.plot([0, 1], [0, 1], ":", color="grey", lw=1)
    ax.set_title(LABELS[d], fontsize=10)
    ax.set_xlabel("FPR (khách tốt bị từ chối)", fontsize=8)
    ax.set_ylabel("TPR (khách xấu bị bắt)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6.5, loc="lower right")
fig.suptitle("Đường ROC trên test (AUC trong ngoặc, tính trên 5 fold gộp)")
fig.tight_layout()
fig.savefig(FIG / "roc_curves.png", dpi=130)
plt.show()

In [ ]:
# --- Đường Precision–Recall với điểm vận hành ---
fig, axes = grid(len(ds_ok))
for ax, d in zip(axes, ds_ok):
    base = SUMMARY[SUMMARY.dataset == d].bad_rate.iloc[0]
    for m in models_ok:
        pr = load_preds(m, d)
        prec, rec, _ = precision_recall_curve(pr.target, pr.proba)
        ax.plot(rec, prec, lw=1.1, color=COLORS[m], label=m)
        s = SUMMARY[(SUMMARY.dataset == d) & (SUMMARY.model == m)].iloc[0]
        ax.plot(s.recall, s.precision, "o", ms=6, color=COLORS[m], markeredgecolor="k")
    ax.axhline(base, color="grey", ls=":", lw=1)
    ax.set_title(LABELS[d], fontsize=10)
    ax.set_xlabel("Recall", fontsize=8)
    ax.set_ylabel("Precision", fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.tick_params(labelsize=7)
axes[0].legend(fontsize=7)
fig.suptitle("Đường precision–recall trên test (chấm tròn: ngưỡng F1 chọn trên validation; "
             "đường chấm: tỷ lệ xấu)")
fig.tight_layout()
fig.savefig(FIG / "pr_curves.png", dpi=130)
plt.show()

In [ ]:
# --- Precision / Recall / F1 tại ngưỡng chọn trên validation ---
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), sharey=True)
w = 0.8 / len(models_ok)
x = np.arange(len(ds_ok))
for ax, (met, title) in zip(axes, [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]):
    for i, m in enumerate(models_ok):
        ax.bar(x + (i - (len(models_ok) - 1) / 2) * w, pivot(met)[m], w,
               yerr=pivot(f"{met}_std")[m].fillna(0), capsize=2, color=COLORS[m], label=m)
    ax.set_xticks(x, [LABELS[d] for d in ds_ok], rotation=30, ha="right", fontsize=8)
    ax.set_title(title)
    ax.set_ylim(0, 1)
axes[0].legend(ncol=4, fontsize=7)
fig.suptitle("Độ đo theo ngưỡng trên test (ngưỡng tối đa F1 trên validation)")
fig.tight_layout()
fig.savefig(FIG / "precision_recall_f1.png", dpi=130)
plt.show()

display(Markdown("**F1 test**"))
display(Markdown(fmt_pm("f1").to_markdown()))
thr = pivot("threshold")
thr.index = [LABELS[d] for d in thr.index]
thr["Tỷ lệ xấu"] = pivot("bad_rate").iloc[:, 0].values
display(Markdown("**Ngưỡng quyết định** (trung bình qua các fold, chọn trên validation)"))
display(thr.round(3))

* Ngưỡng tối ưu F1 thường **thấp hơn 0,5 nhiều** ở các bộ mất cân bằng (GMSC, Home Credit): vì lớp xấu hiếm, xác suất dự đoán hiếm khi vượt 0,5. Dùng ngưỡng mặc định 0,5 sẽ cho recall rất thấp.
* F1 ở GMSC, Home Credit, Lending Club chỉ khoảng 0,2–0,4 dù AUC 0,7–0,86: khi lớp xấu hiếm, bắt được khách xấu mà không từ chối nhầm nhiều khách tốt là bài toán khó.

## 12. Kiểm tra hợp lệ: overfitting và rò rỉ

### 12.1. Overfitting — khoảng cách AUC train và test

* **AUC train − AUC test lớn**: mô hình học thuộc dữ liệu train. RF (cây mọc hết) gần như luôn có AUC train ≈ 1 — đây là đặc tính bình thường của RF, điều quan trọng là AUC test.
* **AUC val − AUC test**: nếu lớn, validation không đại diện cho test (ví dụ trôi dạt thời gian ở Lending Club).

In [ ]:
gap = (pivot("train_auc") - pivot("test_auc_fit_train"))
gap_v = (pivot("val_auc") - pivot("test_auc_fit_train"))
fig, axes = plt.subplots(1, 2, figsize=(15, 0.5 * len(ds_ok) + 1.8))
for ax, g, title in [(axes[0], gap, "AUC train − AUC test (KNN không chấm train)"),
                     (axes[1], gap_v, "AUC val − AUC test")]:
    im = ax.imshow(g.values, cmap="Reds" if g is gap else "RdBu_r",
                   vmin=0 if g is gap else -0.05, vmax=0.5 if g is gap else 0.05, aspect="auto")
    ax.set_xticks(range(len(g.columns)), g.columns)
    ax.set_yticks(range(len(g)), [LABELS[d] for d in g.index])
    ax.grid(False)
    for i in range(g.shape[0]):
        for j in range(g.shape[1]):
            v = g.values[i, j]
            ax.text(j, i, "—" if np.isnan(v) else f"{v:+.3f}", ha="center", va="center", fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(FIG / "overfit_gap.png", dpi=130)
plt.show()

### 12.2. Rò rỉ qua giá trị thiếu ở HMEQ

So sánh AUC của `hmeq` (điền median + nhóm `__missing__`, mô hình *nhìn thấy* ô nào bị thiếu) với `hmeq_blind` (điền ngẫu nhiên, mô hình *không thấy*). Cùng dòng, cùng fold — khác biệt chỉ đến từ thông tin "bị thiếu".

In [ ]:
if {"hmeq", "hmeq_blind"} <= set(ds_ok):
    t = pd.DataFrame({"HMEQ (thấy giá trị thiếu)": pivot("auc").loc["hmeq"],
                      "HMEQ (che giá trị thiếu)": pivot("auc").loc["hmeq_blind"]})
    t["AUC giảm"] = t.iloc[:, 0] - t.iloc[:, 1]
    display(t.round(4))
    ax = t.iloc[:, :2].plot.bar(figsize=(9, 3.6), color=["#C44E52", "#4C72B0"], rot=0)
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel("AUC test")
    ax.set_title("HMEQ: AUC khi mô hình thấy / không thấy mẫu hình thiếu")
    plt.tight_layout()
    plt.savefig(FIG / "hmeq_leakage.png", dpi=130)
    plt.show()
else:
    print("Cần cả hmeq và hmeq_blind")

AUC của các mô hình cây (RF, GBDT) giảm rõ khi che giá trị thiếu: một phần đáng kể "hiệu năng" ở HMEQ đến từ việc học "hồ sơ thiếu thông tin → xấu". Kết luận về HMEQ cần thận trọng; khi xếp hạng chung, có thể thay `hmeq` bằng `hmeq_blind` để kiểm tra độ vững của thứ hạng (mục 13).

## 13. Độ vững của thứ hạng và chi phí tính toán

In [ ]:
# Thứ hạng khi thay HMEQ bằng HMEQ (che thiếu)
if {"hmeq", "hmeq_blind"} <= set(ds_ok):
    alt = [("hmeq_blind" if d == "hmeq" else d) for d in main_ok]
    comp = pd.DataFrame({"Hạng TB (HMEQ gốc)": ranks("auc", main_ok).mean(),
                         "Hạng TB (HMEQ che thiếu)": ranks("auc", alt).mean()}).sort_values("Hạng TB (HMEQ gốc)")
    display(comp.round(2))

In [ ]:
# Thời gian huấn luyện + dự đoán (tổng qua các fold, giây), thang log
tm = FIT.assign(total=FIT.fit_seconds + FIT.predict_seconds).pivot_table(
    index="dataset", columns="model", values="total", aggfunc="sum").loc[ds_ok, models_ok]
fig, ax = plt.subplots(figsize=(13, 4))
w = 0.8 / len(models_ok)
x = np.arange(len(ds_ok))
for i, m in enumerate(models_ok):
    ax.bar(x + (i - (len(models_ok) - 1) / 2) * w, tm[m], w, color=COLORS[m], label=m)
ax.set_yscale("log")
ax.set_xticks(x, [LABELS[d] for d in ds_ok], fontsize=9)
ax.set_ylabel("giây (thang log)")
ax.set_title("Thời gian fit + dự đoán (tổng qua mọi fold)")
ax.legend(ncol=7, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "runtime.png", dpi=130)
plt.show()
display(tm.rename(index=LABELS).round(1))

## 14. Đối chiếu với pipeline chính (`reports/models/comparison`)

Notebook dùng cùng seed, cùng quy tắc chia và cùng tiền xử lý với `src/`, nên AUC phải trùng (hoặc chênh rất nhỏ do khác phiên bản thư viện / số luồng). Ô này chỉ chạy nếu pipeline chính đã được chạy trước đó.

In [ ]:
ref_path = ROOT / "reports" / "models" / "comparison" / "auc.csv"
if ref_path.exists():
    ref = pd.read_csv(ref_path, index_col=0)
    common_d = [d for d in main_ok if d in ref.index]
    common_m = [m for m in models_ok if m in ref.columns]
    diff = (pivot("auc", common_d)[common_m] - ref.loc[common_d, common_m])
    diff.index = [LABELS[d] for d in diff.index]
    display(Markdown("**Chênh AUC: notebook − pipeline chính**"))
    display(diff.style.format("{:+.4f}"))
    print(f"Chênh tuyệt đối lớn nhất: {diff.abs().max().max():.5f}")
else:
    print("Chưa có reports/models/comparison/auc.csv — bỏ qua đối chiếu")

## 15. Tóm tắt tự động

In [ ]:
best = pivot("auc").idxmax(axis=1)
lines = [f"- **{LABELS[d]}**: tốt nhất **{best[d]}** (AUC {pivot('auc').loc[d, best[d]]:.4f}); "
         f"LR {pivot('auc').loc[d, 'LR']:.4f}" for d in ds_ok]
mr = R_AUC.mean().sort_values()
lines += ["", f"**Hạng trung bình theo AUC ({len(main_ok)} bộ chính):** "
          + ", ".join(f"{m} {r:.2f}" for m, r in mr.items()),
          f"**Friedman (Iman–Davenport):** F = {FR['F Iman–Davenport']:.2f}, p = {FR['p (F)']:.4f}; "
          f"Nemenyi CD = {CD:.2f}",
          f"**Khác biệt có ý nghĩa so với {control}:** "
          + (", ".join(VS.loc[VS['p (Holm)'] < 0.05, 'Mô hình']) or "không có")]
display(Markdown("\n".join(lines)))
print("Toàn bộ bảng và hình đã lưu tại:", OUT)

## 16. Kết luận

_(Điền sau khi chạy — xem ô tiếp theo.)_